# update_provenance

**Track 8 — Benchmark & Utility Scripts**

**Description:** Resolve current HEAD commit at runtime — never hardcode.

**Source:** `control/update_provenance.py`

## Paper Linkage

See `specs/PAPER_REVIEW_QUEUE.md` for the full paper → experiment mapping.
This notebook is the `.ipynb` pair of the `.py` control script for `update_provenance`.


In [ ]:
import json
import os
import subprocess
import sys


def _git_commit():
    """Resolve current HEAD commit at runtime — never hardcode."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "unknown"


GIT_COMMIT = _git_commit()

FILES = [
    ("control/pkpd/exp001_baseline.json", "python3 control/pkpd/exp001_hill_dose_response.py"),
    ("control/pkpd/exp002_baseline.json", "python3 control/pkpd/exp002_one_compartment_pk.py"),
    ("control/pkpd/exp003_baseline.json", "python3 control/pkpd/exp003_two_compartment_pk.py"),
    ("control/pkpd/exp004_baseline.json", "python3 control/pkpd/exp004_mab_pk_transfer.py"),
    ("control/pkpd/exp005_baseline.json", "python3 control/pkpd/exp005_population_pk.py"),
    ("control/pkpd/exp006_baseline.json", "python3 control/pkpd/exp006_pbpk_compartments.py"),
    ("control/microbiome/exp010_baseline.json", "python3 control/microbiome/exp010_diversity_indices.py"),
    ("control/microbiome/exp011_baseline.json", "python3 control/microbiome/exp011_anderson_gut_lattice.py"),
    ("control/microbiome/exp012_baseline.json", "python3 control/microbiome/exp012_cdiff_resistance.py"),
    ("control/microbiome/exp013_baseline.json", "python3 control/microbiome/exp013_fmt_rcdi.py"),
    ("control/biosignal/exp020_baseline.json", "python3 control/biosignal/exp020_pan_tompkins_qrs.py"),
    ("control/biosignal/exp021_baseline.json", "python3 control/biosignal/exp021_hrv_metrics.py"),
    ("control/biosignal/exp022_baseline.json", "python3 control/biosignal/exp022_ppg_spo2.py"),
    ("control/biosignal/exp023_baseline.json", "python3 control/biosignal/exp023_fusion.py"),
    ("control/validation/exp040_baseline.json", "python3 control/validation/exp040_barracuda_cpu.py"),
    ("control/endocrine/exp030_baseline.json", "python3 control/endocrine/exp030_testosterone_im_pk.py"),
    ("control/endocrine/exp031_baseline.json", "python3 control/endocrine/exp031_testosterone_pellet_pk.py"),
    ("control/endocrine/exp032_baseline.json", "python3 control/endocrine/exp032_age_testosterone_decline.py"),
    ("control/endocrine/exp033_baseline.json", "python3 control/endocrine/exp033_trt_weight_trajectory.py"),
    ("control/endocrine/exp034_baseline.json", "python3 control/endocrine/exp034_trt_cardiovascular.py"),
    ("control/endocrine/exp035_baseline.json", "python3 control/endocrine/exp035_trt_diabetes.py"),
    ("control/endocrine/exp036_baseline.json", "python3 control/endocrine/exp036_population_trt_montecarlo.py"),
    ("control/endocrine/exp037_baseline.json", "python3 control/endocrine/exp037_testosterone_gut_axis.py"),
    ("control/endocrine/exp038_baseline.json", "python3 control/endocrine/exp038_hrv_trt_cardiovascular.py"),
    ("control/pkpd/exp077_baseline.json", "python3 control/pkpd/exp077_michaelis_menten_pk.py"),
    ("control/microbiome/exp078_baseline.json", "python3 control/microbiome/exp078_antibiotic_perturbation.py"),
    ("control/microbiome/exp079_baseline.json", "python3 control/microbiome/exp079_scfa_production.py"),
    ("control/microbiome/exp080_baseline.json", "python3 control/microbiome/exp080_gut_brain_serotonin.py"),
    ("control/biosignal/exp081_baseline.json", "python3 control/biosignal/exp081_eda_stress.py"),
    ("control/biosignal/exp082_baseline.json", "python3 control/biosignal/exp082_arrhythmia_classification.py"),
    # Track 7: Drug Discovery
    ("control/discovery/exp090_baseline.json", "python3 control/discovery/exp090_matrix_scoring.py"),
    ("control/discovery/exp091_baseline.json", "python3 control/discovery/exp091_addrc_hts.py"),
    ("control/discovery/exp092_baseline.json", "python3 control/discovery/exp092_compound_library.py"),
    ("control/discovery/exp093_baseline.json", "python3 control/discovery/exp093_chembl_jak_panel.py"),
    ("control/discovery/exp094_baseline.json", "python3 control/discovery/exp094_rho_mrtf_fibrosis.py"),
    ("control/discovery/exp097_baseline.json", "python3 control/discovery/exp097_affinity_landscape.py"),
    ("control/toxicology/exp098_baseline.json", "python3 control/toxicology/exp098_toxicity_landscape.py"),
    ("control/toxicology/exp099_baseline.json", "python3 control/toxicology/exp099_hormesis.py"),
    ("control/simulation/exp111_baseline.json", "python3 control/simulation/exp111_causal_terrarium.py"),
    # Track 6: Comparative Medicine
    ("control/comparative/exp100_baseline.json", "python3 control/comparative/exp100_canine_il31.py"),
    ("control/comparative/exp101_baseline.json", "python3 control/comparative/exp101_canine_jak1.py"),
    ("control/comparative/exp102_baseline.json", "python3 control/comparative/exp102_il31_pruritus_timecourse.py"),
    ("control/comparative/exp103_baseline.json", "python3 control/comparative/exp103_lokivetmab_duration.py"),
    ("control/comparative/exp104_baseline.json", "python3 control/comparative/exp104_cross_species_pk.py"),
    ("control/comparative/exp105_baseline.json", "python3 control/comparative/exp105_canine_gut_anderson.py"),
    ("control/comparative/exp106_baseline.json", "python3 control/comparative/exp106_feline_hyperthyroid.py"),
]


def main():
    repo_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    errors = []

    for json_path, command in FILES:
        full_path = f"{repo_root}/{json_path}"
        script = command.replace("python3 ", "")

        try:
            with open(full_path, "r") as f:
                data = json.load(f)

            if "_provenance" not in data:
                errors.append(f"{json_path}: missing _provenance")
                continue

            data["_provenance"]["git_commit"] = GIT_COMMIT
            data["_provenance"]["command"] = command
            data["_provenance"]["script"] = script

            with open(full_path, "w") as f:
                json.dump(data, f, indent=2)
                f.write("\n")

            print(f"Updated {json_path}")
        except Exception as e:
            errors.append(f"{json_path}: {e}")

    if errors:
        for err in errors:
            print(f"ERROR: {err}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()
